# REMORA Upwelling Non-Flat Bathymetry with xamrex

This notebook mirrors the existing non-flat bathymetry example, but reads the AMReX plotfile from `REMORA/Exec/Upwelling` with `xamrex`, including auxiliary multifabs so `h` and `zeta` come directly from the plotfile.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

sys.path.insert(0, "/home/jmsexton/codes/xamrex")
from xamrex import AMReXEntrypoint

In [ ]:
plotfile = Path("/home/jmsexton/codes/REMORA/Exec/Upwelling/plt00010")

ep = AMReXEntrypoint()
ds = ep.open_dataset(
    plotfile,
    include_auxiliary_multifabs=True,
    xroms_format=True,
    xroms_vtransform=2,
)
ds

## Reconstruct the vertical transform

This version uses `h` and `zeta` written into the AMReX plotfile through `remora.plot_vars_2d = h zeta` and loaded with `include_auxiliary_multifabs=True`.

We still need the stretching parameters for the sigma transform, so this notebook uses the REMORA defaults for `theta_s` and `theta_b` and the Upwelling `tcline` value.

Because this setup is periodic in `x`, the bathymetry varies in `eta`/`y`, so the natural section is at fixed `xi`.

In [ ]:
job_info = plotfile / "job_info"


def read_job_info_value(path, key):
    prefix = f"{key} ="
    for line in path.read_text().splitlines():
        if line.startswith(prefix):
            return float(line.split("=", 1)[1].strip())
    raise KeyError(f"Could not find {key!r} in {path}")


tcline = read_job_info_value(job_info, "remora.tcline")
theta_b = read_job_info_value(job_info, "remora.theta_b")
theta_s = read_job_info_value(job_info, "remora.theta_s")
hc = tcline


def remora_stretch(s, theta_s, theta_b):
    if theta_s > 0.0:
        csur = (1.0 - np.cosh(theta_s * s)) / (np.cosh(theta_s) - 1.0)
    else:
        csur = -(s ** 2)

    if theta_b > 0.0:
        return (np.exp(theta_b * csur) - 1.0) / (1.0 - np.exp(-theta_b))
    return csur


def remora_z_r(h, zeta, s_rho, Cs_r, hc):
    cff = hc * s_rho
    cff2 = (cff + Cs_r * h) / (hc + h)
    return zeta + (zeta + h) * cff2


xi_index = ds.sizes["xi_rho"] // 2
h = ds.h.isel(ocean_time=0, xi_rho=xi_index).load()
zeta = ds.zeta.isel(ocean_time=0, xi_rho=xi_index).load()
Cs_r = xr.DataArray(remora_stretch(ds.s_rho.values, theta_s, theta_b), coords={"s_rho": ds.s_rho}, dims=("s_rho",), name="Cs_r")
Cs_w = xr.DataArray(remora_stretch(ds.s_w.values, theta_s, theta_b), coords={"s_w": ds.s_w}, dims=("s_w",), name="Cs_w")
z_r = remora_z_r(h, zeta, ds.s_rho, Cs_r, hc)

xr.Dataset({"h": h, "zeta": zeta, "Cs_r": Cs_r, "Cs_w": Cs_w, "z_r": z_r})

In [ ]:
section = ds.temp.isel(ocean_time=0, xi_rho=xi_index).load()
section = section.assign_coords(
    eta_km=("eta_rho", section.eta_rho.values / 1000.0),
    z=z_r.transpose("s_rho", "eta_rho")
)

fig, ax = plt.subplots(figsize=(10, 4))
mesh = ax.pcolormesh(section.eta_km, section.z, section, shading="auto", cmap="turbo")
ax.plot(section.eta_km, -h, color="k", lw=1.5)
ax.set_xlabel("y (km)")
ax.set_ylabel("z (m)")
ax.set_title(f"Temperature section at xi index {xi_index}")
fig.colorbar(mesh, ax=ax, label="temperature")
plt.show()

In [ ]:
pl = section.plot(x="eta_km", y="z", figsize=(10, 4), cmap="turbo")
pl.axes.plot(section.eta_km, -h, color="k", lw=1.5)
pl.axes.set_xlabel("y (km)")
pl.axes.set_ylabel("z (m)")
pl.axes.set_title("Same section with xarray plotting")